### RAG pipleline - Data Ingestion to Vector DB PileLine

In [49]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import tqdm as notebook_tqdm
import os

In [50]:
def load_pdf_documents(pdf_directory: str) -> list[Document]:
    all_docs = []
    pdf_path = Path(pdf_directory)

    pdf_files = list(pdf_path.glob("**/*.pdf"))
    print(f"I found {len(pdf_files)}")
    for pdf_file in pdf_files:
        print(f"\n Processing :{pdf_file.name}")
        try:
            loader = PyMuPDFLoader(pdf_file)
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_docs.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error : {e}")
    return all_docs

In [51]:
all_pdf_docs = load_pdf_documents("data/pdf_files")


I found 3

 Processing :AI engineering Interview.pdf
loaded 20 pages

 Processing :HR interview.pdf
loaded 12 pages

 Processing :Type script interview.pdf
loaded 20 pages


### Text splitting get to chunks


In [52]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content : {split_docs[0].page_content[:200]} ...")
        print(f"Metadata : {split_docs[0].metadata} ...")
    return split_docs

In [53]:
chunks = split_documents(all_pdf_docs, chunk_size=1000, chunk_overlap=200)

Split 52 documents into 126 chunks

 Example chunk:
Content : Absolutely. Below is a 100-question AI Engineering interview set based directly on the skills you provided. I also added 
dedicated questions about LangGraph and smolagents, especially around agents,  ...
Metadata : {'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data\\pdf_files\\AI engineering Interview.pdf', 'file_path': 'data\\pdf_files\\AI engineering Interview.pdf', 'total_pages': 20, 'format': 'PDF 1.4', 'title': 'AI engineering Interview', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'AI engineering Interview.pdf', 'file_type': 'pdf'} ...


### Embedding and vectorStoreDB

In [54]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformers"""

    def __init__(self, model_name: str = "all-MiniLm-L6-v2"):
        """Initirialize the embedding manager

        Args:
            model_name (str, optional): _description_. HuggingFace model name for sentence embeddings Defaults to "all-miniLm-L6-v2".
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        "Load the SentenceTransformer model"
        try:
            print(f"Loading embedding models: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfuly. Embedding dimension:{ self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading models {self._model}: {e}")
            raise

    def generate_embedding(self, texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts (List[str]): List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimansion(self)->int:
        """Get the embedding dimension of the model"""  
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding models: all-MiniLm-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2709.04it/s]


Model loaded successfuly. Embedding dimension:384


C:\Users\Me\AppData\Local\Temp\ipykernel_4044\126679937.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfuly. Embedding dimension:{ self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [63]:
class VactoreStore:
    """manages document embedding in a chromaDB vector store
    """
    def __init__(self, collection_name: str = "pdf_document", persist_directory: str= "data/vector_store"):
        """initialize the vector store

        Args:
            collection_name (str, optional): Name of the ChromaDB collection. Defaults to "pdf_document".
            persist_directory (str, optional): Directory to persist the vector store. Defaults to "data/vector_store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        """
        Initialze ChromaDB client and collection
        """
        try:
            # Create persistent Chroma DB
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for RAG"}
            )
            print(f"Vector store initiazed.Collection: {self.collection_name}")
            print(f"Exiting document is colection:{self.collection.count()}")
        except Exception as e:
            print(f"Error initialize vector store: {e}")
            raise
    def add_document(self, documents: List[any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents (List[any]): List of langchain documents
            embeddings (np.ndarray): Corresponding embeddings for documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must math unmber of embeddings")
        print(f"Adding {len(documents)} documents to vector store ...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['contenct_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)

            embeddings_list.append(embedding)
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfuly added {len(documents)} documents to vector store")
            print(f"total document in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding document to vector sore: {e}")
            raise
vectore_store = VactoreStore()
vectore_store

Vector store initiazed.Collection: pdf_document
Exiting document is colection:0


### convert texts to embeddings

In [64]:
texts = [doc.page_content for doc in chunks]



### Generate the embeddings

In [65]:
embeddings = embedding_manager.generate_embedding(texts=texts)

### store in the database

vectore_store.add_document(documents=chunks, embeddings=embeddings)

Generating embeddings for 126 texts ...


Batches: 100%|██████████| 4/4 [00:06<00:00,  1.72s/it]


Generated embeddings with shape: (126, 384)
Adding 126 documents to vector store ...
Successfuly added 126 documents to vector store
total document in collection: 126
